# Punta Arenas — gas subsidy and heat-transition analysis

Dedicated analysis notebook for the paper on replacing natural-gas end uses in Punta Arenas/Magallanes.

This notebook deliberately separates **data preparation** from **paper analysis**:

1. Process raw gas Excel files in standalone scripts.
2. Load the resulting hourly gas/sector files here.
3. Convert gas volumes to energy only with an explicit, documented conversion factor.
4. Load useful-heat demand profiles by end use.
5. Quantify the gas-subsidy baseline and avoided subsidy under transition scenarios.
6. Load PyPSA results with the heat sector enabled and compare heat supply, electricity impacts, gas displacement, emissions, capacities, and costs.
7. Export paper-ready tables and figures.

The structure follows the newer `case_analysis_PuntaArenas.ipynb` (v7) workflow, while reusing and generalising the heat-sector extraction logic that existed only in `case_analysis_PuntaArenas_updated_results.ipynb`.


## 1. Imports and analysis controls

Keep numerical assumptions in this cell so that every paper figure can be traced back to one explicit value or input file.


In [ ]:
from pathlib import Path
import importlib
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

try:
    import pypsa
except ImportError:
    pypsa = None

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

# -----------------------------------------------------------------------------
# Study controls
# -----------------------------------------------------------------------------
CASE_FOLDER = Path("input_punta_arenas")
ANALYSIS_YEAR = 2025
CITY = "punta_arenas"

OUTPUT_ROOT = Path("output") / "output_punta_arenas_heat_paper"
TABLE_DIR = OUTPUT_ROOT / "tables"
FIGURE_DIR = OUTPUT_ROOT / "figures"
MODEL_EXPORT_DIR = OUTPUT_ROOT / "simulations"
for folder in (OUTPUT_ROOT, TABLE_DIR, FIGURE_DIR, MODEL_EXPORT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

PAPER_DPI = 300
SAVE_FIGURES = True

# -----------------------------------------------------------------------------
# Gas-energy conversion
# -----------------------------------------------------------------------------
# IMPORTANT: leave as None until a source/utility convention is confirmed.
# Once confirmed, write the value and source in GAS_LHV_SOURCE below.
GAS_LHV_KWH_PER_M3 = None
GAS_LHV_SOURCE = "PENDING: insert documented lower-heating-value / standard-m3 convention"

# Optional policy/economic conversion.
CLP_PER_USD = None  # set only if a figure/table needs USD conversion


# -----------------------------------------------------------------------------
# Gas-subsidy accounting
# -----------------------------------------------------------------------------
# IMPORTANT: the annual regional figures below are validation benchmarks only.
# They are NOT applied directly to Punta Arenas scenario gas consumption.
OFFICIAL_SUBSIDY_BENCHMARK_2025_CLP = 65_629_000_000  # 2025 budget allocation
OFFICIAL_SUBSIDY_ACTUAL_2025_USD = 64_600_000         # ENAP 2025 reported compensation

OFFICIAL_SUBSIDY_SOURCES = {
    "budget_2025": "Ministerio de Energía / Cuenta Pública Magallanes 2025: CLP 65.629 billion",
    "actual_2025": "ENAP 2025 financial analysis: USD 64.6 million",
    "mechanism": "CNE / Ley de Presupuestos: monthly compensation based on eligible gas volumes and the production/contract/factured price gaps",
}

# A unit subsidy factor must come from the documented policy calculation/table.
# Do not infer it from the annual regional total unless the corresponding eligible
# regional gas volume is also known on the same basis.
SUBSIDY_CLP_PER_M3_DEFAULT = None
SUBSIDY_CLP_PER_MWH_DEFAULT = None

# Gas-fired power technologies that may convert heat electrification into
# additional subsidised gas consumption in the electricity sector.
GAS_POWER_CARRIERS = {
    "gas", "natural_gas", "ocgt", "ccgt", "gas_turbine",
    "gas_engine", "gas_generator", "thermal_gas",
}

# Technologies to track in model results. Extra carriers are harmless if absent.
HEAT_TECH_CARRIERS = [
    "gas_boiler",
    "heat_pump",
    "electric_boiler",
    "biogas_boiler",
    "district_heating",
    "biomass_boiler",
    "pellet_boiler",
]

TECH_LABELS = {
    "gas_boiler": "Natural-gas boiler",
    "heat_pump": "Heat pump",
    "electric_boiler": "Electric boiler",
    "biogas_boiler": "Biogas boiler",
    "district_heating": "District heating",
    "biomass_boiler": "Biomass boiler",
    "pellet_boiler": "Pellet boiler",
}


## 2. Input file discovery

The notebook first looks for the current outputs of the gas-processing workflow. Paths can be changed here without touching the analysis cells.


In [ ]:
def first_existing(candidates, required=False, label="input"):
    candidates = [Path(p) for p in candidates]
    for path in candidates:
        if path.exists():
            return path
    if required:
        raise FileNotFoundError(
            f"Could not find {label}. Tried:\n" + "\n".join(f"  - {p}" for p in candidates)
        )
    return None

GAS_HOURLY_SECTOR_FILE = first_existing([
    CASE_FOLDER / "gas" / "consumo_gn_horario_sector_mt3.csv",
    CASE_FOLDER / "gas_demand" / "consumo_gn_horario_sector_mt3.csv",
    Path("gas_demand") / "consumo_gn_horario_sector_mt3.csv",
    Path("consumo_gn_horario_sector_mt3.csv"),
], label="hourly gas consumption by sector")

GAS_HOURLY_TOTAL_FILE = first_existing([
    CASE_FOLDER / "gas" / "consumo_gn_horario_total.csv",
    CASE_FOLDER / "gas_demand" / "consumo_gn_horario_total.csv",
    Path("gas_demand") / "consumo_gn_horario_total.csv",
    Path("consumo_gn_horario_total.csv"),
], label="hourly total gas consumption")

GAS_DAILY_COMPANIES_FILE = first_existing([
    CASE_FOLDER / "gas" / "consumo_gn_diario_empresas.csv",
    CASE_FOLDER / "gas_demand" / "consumo_gn_diario_empresas.csv",
    Path("gas_demand") / "consumo_gn_diario_empresas.csv",
    Path("consumo_gn_diario_empresas.csv"),
], label="daily historical gas consumption")

HEAT_PROFILE_FILE = first_existing([
    CASE_FOLDER / "demand_profiles_heat.csv",
    CASE_FOLDER / "demand_profiles" / "demand_profiles_heat.csv",
    CASE_FOLDER / "demand_profiles_scenarios" / "demand_profiles_heat.csv",
    Path("demand_profiles_heat.csv"),
    Path("heat_demand_profiles.csv"),
], label="useful-heat demand profile")

SUBSIDY_FILE = first_existing([
    CASE_FOLDER / "gas_subsidy.csv",
    CASE_FOLDER / "gas" / "gas_subsidy.csv",
    Path("gas_subsidy.csv"),
], label="gas subsidy assumptions")

HEAT_ASSETS_FILE = first_existing([
    CASE_FOLDER / "heat_assets.csv",
    Path("heat_assets.csv"),
], label="heat assets")

COSTS_FILE = first_existing([
    CASE_FOLDER / "costs.csv",
    Path("costs.csv"),
], label="technology costs")

input_catalog = pd.DataFrame({
    "input": [
        "gas_hourly_sector", "gas_hourly_total", "gas_daily_companies",
        "heat_profiles", "gas_subsidy", "heat_assets", "costs",
    ],
    "path": [
        GAS_HOURLY_SECTOR_FILE, GAS_HOURLY_TOTAL_FILE, GAS_DAILY_COMPANIES_FILE,
        HEAT_PROFILE_FILE, SUBSIDY_FILE, HEAT_ASSETS_FILE, COSTS_FILE,
    ],
})
input_catalog["available"] = input_catalog["path"].notna()
display(input_catalog)


## 3. Gas-consumption data: load and validate

Expected current sectoral output columns are `timestamp`, `comuna`, `sector`, and `consumo_horario_sector`. The processing workflow labels this quantity in m³; this notebook checks structure and conservation before any energy conversion.


In [ ]:
def load_hourly_sector_gas(path):
    if path is None:
        return pd.DataFrame()
    df = pd.read_csv(path)
    if "timestamp" not in df.columns:
        raise ValueError(f"{path}: missing 'timestamp' column")
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

    rename = {}
    if "consumo_horario_sector" in df.columns:
        rename["consumo_horario_sector"] = "gas_m3"
    elif "value_m3" in df.columns:
        rename["value_m3"] = "gas_m3"
    elif "gas_m3" not in df.columns:
        raise ValueError(
            f"{path}: expected 'consumo_horario_sector', 'value_m3', or 'gas_m3'. "
            f"Columns are: {list(df.columns)}"
        )
    df = df.rename(columns=rename)

    if "comuna" in df.columns:
        df["comuna"] = df["comuna"].astype(str).str.strip().str.lower()
    if "sector" in df.columns:
        df["sector"] = df["sector"].astype(str).str.strip()
    df["gas_m3"] = pd.to_numeric(df["gas_m3"], errors="coerce")
    return df.dropna(subset=["timestamp", "gas_m3"]).sort_values("timestamp")


gas_sector = load_hourly_sector_gas(GAS_HOURLY_SECTOR_FILE)

if gas_sector.empty:
    print("Hourly sectoral gas file not found yet. Run the gas preprocessing workflow first.")
else:
    gas_city = gas_sector[gas_sector.get("comuna", CITY).eq(CITY)].copy() if "comuna" in gas_sector.columns else gas_sector.copy()
    print(f"Rows: {len(gas_city):,}")
    print(f"Period: {gas_city['timestamp'].min()} -> {gas_city['timestamp'].max()}")
    print(f"Negative values: {(gas_city['gas_m3'] < 0).sum():,}")
    print(f"Missing gas values: {gas_city['gas_m3'].isna().sum():,}")
    if "sector" in gas_city.columns:
        print("Sectors:", sorted(gas_city["sector"].dropna().unique()))
    display(gas_city.head())


In [ ]:
# Conservation / completeness diagnostics
if not gas_sector.empty:
    gas_city = gas_sector[gas_sector["comuna"].eq(CITY)].copy() if "comuna" in gas_sector.columns else gas_sector.copy()
    gas_city["date"] = gas_city["timestamp"].dt.floor("D")
    gas_city["year"] = gas_city["timestamp"].dt.year
    gas_city["month"] = gas_city["timestamp"].dt.month
    gas_city["hour"] = gas_city["timestamp"].dt.hour

    hourly_coverage = (
        gas_city[["timestamp"]].drop_duplicates()
        .assign(date=lambda x: x["timestamp"].dt.floor("D"))
        .groupby("date").size()
    )
    bad_days = hourly_coverage[hourly_coverage != 24]

    print(f"Unique hourly timestamps: {gas_city['timestamp'].nunique():,}")
    print(f"Days not containing exactly 24 hourly timestamps: {len(bad_days):,}")
    if len(bad_days):
        display(bad_days.head(20).rename("n_hours").to_frame())

    annual_gas = (
        gas_city.groupby(["year", "sector"], as_index=False)["gas_m3"].sum()
        if "sector" in gas_city.columns
        else gas_city.groupby("year", as_index=False)["gas_m3"].sum()
    )
    display(annual_gas)


## 4. Historical gas-consumption context

This section uses the longer GMAG/EDELMAG daily series when available. It is descriptive and independent from the model-year calibration.


In [ ]:
if GAS_DAILY_COMPANIES_FILE is None:
    gas_daily = pd.DataFrame()
    print("Historical daily GMAG/EDELMAG file not found.")
else:
    gas_daily = pd.read_csv(GAS_DAILY_COMPANIES_FILE)
    date_col = "fecha" if "fecha" in gas_daily.columns else "date"
    gas_daily[date_col] = pd.to_datetime(gas_daily[date_col], errors="coerce")
    city_col = CITY if CITY in gas_daily.columns else None

    if city_col:
        series = gas_daily.dropna(subset=[date_col, city_col]).copy()
        series[city_col] = pd.to_numeric(series[city_col], errors="coerce")
        series["year"] = series[date_col].dt.year
        annual = series.groupby(["year", "empresa"], as_index=False)[city_col].sum()
        display(annual.tail(20))

        fig, ax = plt.subplots(figsize=(8, 4.2))
        for company, sub in annual.groupby("empresa"):
            ax.plot(sub["year"], sub[city_col], marker="o", label=str(company).upper())
        ax.set_ylabel("Reported gas consumption [source units/yr]")
        ax.set_xlabel("Year")
        ax.set_title("Punta Arenas historical gas consumption by company")
        ax.legend()
        ax.grid(True, alpha=0.25)
        plt.tight_layout()
        if SAVE_FIGURES:
            fig.savefig(FIGURE_DIR / "gas_historical_companies.png", dpi=PAPER_DPI, bbox_inches="tight")
        plt.show()


## 5. Convert gas volume to fuel energy — only after the unit convention is confirmed

No default calorific value is silently assumed. When `GAS_LHV_KWH_PER_M3` is documented, the notebook derives `gas_mwh_fuel` and records the source in exported metadata.


In [ ]:
def add_gas_energy(df, volume_col="gas_m3"):
    out = df.copy()
    if out.empty:
        return out
    if GAS_LHV_KWH_PER_M3 is None:
        out["gas_mwh_fuel"] = np.nan
        print(
            "GAS_LHV_KWH_PER_M3 is still None: volume data are retained in m3 and no energy conversion is performed."
        )
    else:
        out["gas_mwh_fuel"] = out[volume_col] * float(GAS_LHV_KWH_PER_M3) / 1000.0
    return out

if not gas_sector.empty:
    gas_sector = add_gas_energy(gas_sector)
    metadata = {
        "gas_lhv_kwh_per_m3": GAS_LHV_KWH_PER_M3,
        "gas_lhv_source": GAS_LHV_SOURCE,
    }
    print(metadata)


## 6. Gas profiles by sector

These plots describe the observed gas-demand structure before translating it into useful heat by end use.


In [ ]:
if not gas_sector.empty and "sector" in gas_sector.columns:
    gas_city = gas_sector[gas_sector["comuna"].eq(CITY)].copy() if "comuna" in gas_sector.columns else gas_sector.copy()
    gas_city["month"] = gas_city["timestamp"].dt.month
    gas_city["hour"] = gas_city["timestamp"].dt.hour

    monthly_sector = gas_city.groupby(["month", "sector"])["gas_m3"].sum().unstack(fill_value=0)
    hourly_shape = gas_city.groupby(["hour", "sector"])["gas_m3"].mean().unstack(fill_value=0)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    monthly_sector.plot(kind="bar", stacked=True, ax=axes[0])
    axes[0].set_xlabel("Month")
    axes[0].set_ylabel("Gas consumption [m3/month]")
    axes[0].set_title("Observed gas consumption by sector")
    axes[0].legend(fontsize=8)

    hourly_shape.plot(ax=axes[1])
    axes[1].set_xlabel("Hour of day")
    axes[1].set_ylabel("Average gas consumption [m3/h]")
    axes[1].set_title("Average hourly gas profile by sector")
    axes[1].legend(fontsize=8)
    for ax in axes:
        ax.grid(True, alpha=0.25)
    plt.tight_layout()
    if SAVE_FIGURES:
        fig.savefig(FIGURE_DIR / "gas_sector_profiles.png", dpi=PAPER_DPI, bbox_inches="tight")
    plt.show()


## 7. Useful-heat demand profiles by end use

Preferred long-format schema:

`timestamp, zone, end_use, demand_mw_th`

Aliases such as `demand_type` and `demand_mw` are accepted. The heat-profile builder should perform the gas-to-useful-heat allocation; this notebook validates and analyses the result rather than reproducing that ETL logic.


In [ ]:
def load_heat_profiles(path):
    if path is None:
        return pd.DataFrame()
    df = pd.read_csv(path)
    if "timestamp" not in df.columns:
        raise ValueError(f"{path}: missing timestamp")
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

    if "end_use" not in df.columns and "demand_type" in df.columns:
        df = df.rename(columns={"demand_type": "end_use"})
    if "demand_mw_th" not in df.columns:
        if "demand_mw" in df.columns:
            df = df.rename(columns={"demand_mw": "demand_mw_th"})
        elif "heat_mw" in df.columns:
            df = df.rename(columns={"heat_mw": "demand_mw_th"})
        else:
            raise ValueError(f"{path}: expected demand_mw_th, demand_mw, or heat_mw")

    if "zone" not in df.columns:
        df["zone"] = CITY
    if "end_use" not in df.columns:
        df["end_use"] = "heat_total"

    df["demand_mw_th"] = pd.to_numeric(df["demand_mw_th"], errors="coerce")
    return df.dropna(subset=["timestamp", "demand_mw_th"]).sort_values("timestamp")

heat_profiles = load_heat_profiles(HEAT_PROFILE_FILE)
if heat_profiles.empty:
    print("Useful-heat profile file not available yet; this section will populate once the end-use builder is finished.")
else:
    display(heat_profiles.head())
    print("End uses:", sorted(heat_profiles["end_use"].astype(str).unique()))
    print("Zones:", sorted(heat_profiles["zone"].astype(str).unique()))


In [ ]:
if not heat_profiles.empty:
    hp = heat_profiles.copy()
    hp["year"] = hp["timestamp"].dt.year
    hp["month"] = hp["timestamp"].dt.month
    hp["hour"] = hp["timestamp"].dt.hour

    hp_y = hp[hp["year"].eq(ANALYSIS_YEAR)].copy()
    if hp_y.empty:
        hp_y = hp.copy()
        print(f"No rows for {ANALYSIS_YEAR}; showing all available years.")

    annual_by_enduse = hp_y.groupby("end_use")["demand_mw_th"].sum().sort_values(ascending=False)
    monthly_by_enduse = hp_y.groupby(["month", "end_use"])["demand_mw_th"].sum().unstack(fill_value=0)
    hourly_by_enduse = hp_y.groupby(["hour", "end_use"])["demand_mw_th"].mean().unstack(fill_value=0)

    display(annual_by_enduse.rename("MWh_th/year").to_frame())

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    annual_by_enduse.plot(kind="bar", ax=axes[0])
    axes[0].set_ylabel("Useful heat [MWh_th/yr]")
    axes[0].set_title("Annual useful-heat demand")

    monthly_by_enduse.plot(kind="bar", stacked=True, ax=axes[1], legend=False)
    axes[1].set_ylabel("Useful heat [MWh_th/month]")
    axes[1].set_title("Seasonal heat demand")

    hourly_by_enduse.plot(ax=axes[2])
    axes[2].set_ylabel("Average heat demand [MW_th]")
    axes[2].set_title("Average hourly profile")
    axes[2].legend(fontsize=8)
    for ax in axes:
        ax.grid(True, alpha=0.25)
    plt.tight_layout()
    if SAVE_FIGURES:
        fig.savefig(FIGURE_DIR / "heat_demand_end_use_profiles.png", dpi=PAPER_DPI, bbox_inches="tight")
    plt.show()


## 8. Gas-to-heat reconciliation

This is the key consistency check between the gas dataset and the end-use heat profiles. The exact comparison depends on whether the heat builder represents **fuel input** or **useful heat output**. Keep that convention explicit.


In [ ]:
def annual_heat_reconciliation(gas_df, heat_df, year=ANALYSIS_YEAR, city=CITY):
    rows = []

    if not gas_df.empty:
        g = gas_df.copy()
        if "comuna" in g.columns:
            g = g[g["comuna"].eq(city)]
        g = g[g["timestamp"].dt.year.eq(year)]
        rows.append({
            "quantity": "Observed gas volume",
            "value": g["gas_m3"].sum(),
            "unit": "m3/yr",
        })
        if "gas_mwh_fuel" in g.columns and g["gas_mwh_fuel"].notna().any():
            rows.append({
                "quantity": "Observed gas fuel energy",
                "value": g["gas_mwh_fuel"].sum(),
                "unit": "MWh_fuel/yr",
            })

    if not heat_df.empty:
        h = heat_df[heat_df["timestamp"].dt.year.eq(year)].copy()
        rows.append({
            "quantity": "Reconstructed useful heat",
            "value": h["demand_mw_th"].sum(),
            "unit": "MWh_th/yr",
        })

    return pd.DataFrame(rows)

reconciliation = annual_heat_reconciliation(gas_sector, heat_profiles)
display(reconciliation)


## 9. Gas subsidy: policy basis and accounting convention

For this paper, the gas subsidy is treated as a **system-level policy cost**, not only as a household bill discount.

The 2025 Chilean budget defines the *Aporte Compensatorio* as a transfer to ENAP for the lower value obtained from gas sales to the Magallanes distributor for consumers subject to the guaranteed tariff. The amount is determined monthly by the CNE using eligible gas volumes and the relevant production / contract / invoiced-price gaps.

Two 2025 regional benchmarks are retained only for validation:

- **CLP 65.629 billion**: 2025 budget allocation for the Magallanes gas compensation.
- **USD 64.6 million**: compensation reported by ENAP for 2025.

These regional totals must **not** be multiplied or scaled directly into Punta Arenas scenarios. Scenario accounting should use a documented subsidy intensity (CLP/m³ or CLP/MWh-fuel) on the gas volumes that are actually eligible.

Most importantly, electrification can reduce direct gas consumption while increasing electricity demand. If part of that electricity is produced with subsidised gas, the relevant quantity is therefore:

**net eligible gas = direct end-use gas + gas used for power generation**

and the main policy metric is:

**avoided subsidy = baseline net subsidy − scenario net subsidy**


In [ ]:
def load_subsidy_assumptions(path):
    if path is None:
        return pd.DataFrame()
    df = pd.read_csv(path)
    if "year" in df.columns:
        df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
    return df

subsidy = load_subsidy_assumptions(SUBSIDY_FILE)

official_subsidy_benchmarks = pd.DataFrame([
    {
        "year": 2025,
        "metric": "Budget allocation",
        "value": OFFICIAL_SUBSIDY_BENCHMARK_2025_CLP,
        "unit": "CLP/year",
        "scope": "Magallanes regional benchmark",
        "use_in_model": False,
    },
    {
        "year": 2025,
        "metric": "ENAP reported compensation",
        "value": OFFICIAL_SUBSIDY_ACTUAL_2025_USD,
        "unit": "USD/year",
        "scope": "Magallanes regional benchmark",
        "use_in_model": False,
    },
])

display(official_subsidy_benchmarks)

if subsidy.empty:
    print(
        "No gas_subsidy.csv found yet. Scenario subsidy values will remain unavailable "
        "until a documented CLP/m3 or CLP/MWh-fuel factor (preferably monthly) is provided."
    )
else:
    display(subsidy)


In [ ]:
def calculate_subsidy(gas_df, subsidy_df, year=ANALYSIS_YEAR, city=CITY):
    """Estimate observed direct-gas subsidy for validation.

    Supported policy inputs:
      - subsidy_clp_per_m3
      - or reference_price_clp_per_mwh and paid_price_clp_per_mwh

    This function describes direct observed gas only. Whole-system scenario accounting,
    including gas-fired electricity generation, is done later in the notebook.
    """
    if gas_df.empty or subsidy_df.empty:
        return pd.DataFrame()

    g = gas_df.copy()
    if "comuna" in g.columns:
        g = g[g["comuna"].eq(city)]
    g = g[g["timestamp"].dt.year.eq(year)].copy()
    if g.empty:
        return pd.DataFrame()

    if "sector" in g.columns:
        consumption = g.groupby("sector", as_index=False).agg(
            gas_m3=("gas_m3", "sum"),
            gas_mwh_fuel=("gas_mwh_fuel", "sum") if "gas_mwh_fuel" in g.columns else ("gas_m3", lambda x: np.nan),
        )
    else:
        consumption = pd.DataFrame({
            "sector": ["total"],
            "gas_m3": [g["gas_m3"].sum()],
            "gas_mwh_fuel": [g["gas_mwh_fuel"].sum() if "gas_mwh_fuel" in g.columns else np.nan],
        })

    s = subsidy_df.copy()
    if "year" in s.columns:
        s = s[s["year"].eq(year)]
    if "sector" not in s.columns:
        s["sector"] = "total"

    if set(s["sector"].astype(str)) == {"total"} and len(consumption) > 1:
        row = s.iloc[0].to_dict()
        s = pd.DataFrame([{**row, "sector": sec} for sec in consumption["sector"]])

    out = consumption.merge(s, on="sector", how="left")
    out["subsidy_clp"] = np.nan

    direct = out.get("subsidy_clp_per_m3", pd.Series(np.nan, index=out.index)).notna()
    out.loc[direct, "subsidy_clp"] = (
        out.loc[direct, "gas_m3"] * out.loc[direct, "subsidy_clp_per_m3"]
    )

    price_cols = {"reference_price_clp_per_mwh", "paid_price_clp_per_mwh"}
    if price_cols.issubset(out.columns):
        implicit = (~direct) & out["gas_mwh_fuel"].notna()
        out.loc[implicit, "subsidy_clp"] = (
            out.loc[implicit, "gas_mwh_fuel"]
            * (out.loc[implicit, "reference_price_clp_per_mwh"] - out.loc[implicit, "paid_price_clp_per_mwh"])
        )

    if CLP_PER_USD:
        out["subsidy_usd"] = out["subsidy_clp"] / float(CLP_PER_USD)
    return out

subsidy_baseline = calculate_subsidy(gas_sector, subsidy)
if not subsidy_baseline.empty:
    display(subsidy_baseline)
    print(f"Observed direct-gas subsidy estimate: {subsidy_baseline['subsidy_clp'].sum():,.0f} CLP/yr")


## 10. Heat technology assumptions and references

This cell is designed to make the paper auditable: technology availability, efficiency/COP, lifetime, CAPEX/OPEX, and source/reference columns should be visible before running or interpreting scenarios.


In [ ]:
def read_csv_optional(path):
    return pd.DataFrame() if path is None else pd.read_csv(path)

heat_assets = read_csv_optional(HEAT_ASSETS_FILE)
costs = read_csv_optional(COSTS_FILE)

if heat_assets.empty:
    print("heat_assets.csv not found yet.")
else:
    print("heat_assets.csv")
    display(heat_assets)

if costs.empty:
    print("costs.csv not found yet.")
else:
    heat_cost_mask = pd.Series(False, index=costs.index)
    for col in ["technology", "cost_key", "carrier"]:
        if col in costs.columns:
            heat_cost_mask |= costs[col].astype(str).str.lower().str.contains(
                "heat|boiler|pump|biogas|biomass|pellet|district|natural_gas", regex=True, na=False
            )
    heat_costs = costs.loc[heat_cost_mask].copy()
    preferred_cols = [c for c in [
        "cost_key", "technology", "year", "capex", "capital_cost", "marginal_cost",
        "efficiency", "lifetime_years", "source", "source_url", "notes"
    ] if c in heat_costs.columns]
    display(heat_costs[preferred_cols] if preferred_cols else heat_costs)


## 11. Scenario registry

The analysis notebook should not hard-code technology implementation details. Each scenario points to exported model results. When the heat-technology representation is finalised, keep scenario-specific CSVs/build logic in the model inputs and only register the scenario code and label here.


In [ ]:
SCENARIOS = {
    "GAS_BASE": "Gas baseline",
    "HP": "Heat pumps",
    "EB": "Electric boilers",
    "BIOGAS": "Biogas boilers",
    "DH": "District heating",
    "BIOMASS": "Biomass / pellets",
}

# Only scenarios with existing exports will be loaded.
# Expected layout (same paper-analysis principle as Punta Arenas v7):
# output/output_punta_arenas_heat_paper/simulations/<scenario>/<year>/network.nc

def network_path_for(scenario, year):
    return MODEL_EXPORT_DIR / str(scenario) / str(year) / "network.nc"

available_networks = []
for scenario, label in SCENARIOS.items():
    path = network_path_for(scenario, ANALYSIS_YEAR)
    available_networks.append({
        "scenario": scenario,
        "label": label,
        "year": ANALYSIS_YEAR,
        "network_path": path,
        "available": path.exists(),
    })
available_networks = pd.DataFrame(available_networks)
display(available_networks)


## 12. Load exported PyPSA networks

The preferred workflow is: **solve once → export network/results → analyse repeatedly**. This avoids re-running optimisation whenever a plot changes.


In [ ]:
def load_exported_networks(registry, year=ANALYSIS_YEAR):
    networks = {}
    if pypsa is None:
        print("PyPSA is not installed in this environment; network loading skipped.")
        return networks
    for scenario in registry:
        path = network_path_for(scenario, year)
        if path.exists():
            networks[scenario] = pypsa.Network(path)
    return networks

networks = load_exported_networks(SCENARIOS)
print("Loaded scenarios:", list(networks))


## 13. Generic heat-sector extraction

This generalises the heat analysis from the older Punta Arenas notebook. It works with any heat-supply technology represented as a PyPSA `Link`, not only gas boilers and heat pumps.


In [ ]:
def snapshot_weights(n):
    if hasattr(n, "snapshot_weightings") and "objective" in n.snapshot_weightings:
        return pd.Series(n.snapshot_weightings["objective"], index=n.snapshots).astype(float)
    return pd.Series(1.0, index=n.snapshots)


def link_input_series(n, links):
    links = [x for x in links if x in n.links.index]
    if not links:
        return pd.Series(0.0, index=n.snapshots)
    return n.links_t.p0.reindex(index=n.snapshots, columns=links, fill_value=0.0).clip(lower=0).sum(axis=1)


def link_output_series(n, links):
    links = [x for x in links if x in n.links.index]
    if not links:
        return pd.Series(0.0, index=n.snapshots)
    if hasattr(n.links_t, "p1") and not n.links_t.p1.empty:
        return n.links_t.p1.reindex(index=n.snapshots, columns=links, fill_value=0.0).abs().sum(axis=1)
    eff = n.links.loc[links, "efficiency"].fillna(1.0)
    p0 = n.links_t.p0.reindex(index=n.snapshots, columns=links, fill_value=0.0).clip(lower=0)
    return p0.multiply(eff, axis=1).sum(axis=1)


def heat_links_by_carrier(n):
    if n.links.empty or "carrier" not in n.links.columns:
        return {}
    groups = {}
    for carrier in HEAT_TECH_CARRIERS:
        idx = n.links.index[n.links["carrier"].astype(str).eq(carrier)].tolist()
        if idx:
            groups[carrier] = idx
    return groups


def heat_operation(n):
    groups = heat_links_by_carrier(n)
    rows = []
    timeseries = {}
    w = snapshot_weights(n)

    for carrier, links in groups.items():
        inp = link_input_series(n, links)
        out = link_output_series(n, links)
        timeseries[carrier] = {"input": inp, "heat_output": out}

        cap_col = "p_nom_opt" if "p_nom_opt" in n.links.columns else "p_nom"
        capacity = pd.to_numeric(n.links.loc[links, cap_col], errors="coerce").fillna(0).sum()
        rows.append({
            "carrier": carrier,
            "technology": TECH_LABELS.get(carrier, carrier),
            "input_mwh": float((inp * w).sum()),
            "heat_output_mwh": float((out * w).sum()),
            "capacity_mw_input": float(capacity),
        })

    return pd.DataFrame(rows), timeseries


## 14. Cross-scenario heat results

Core paper metrics: useful heat supplied, gas use, additional electricity consumption, installed heat capacity, and technology shares.


In [ ]:
scenario_heat_rows = []
scenario_timeseries = {}

for scenario, n in networks.items():
    summary, ts = heat_operation(n)
    scenario_timeseries[scenario] = ts
    if summary.empty:
        continue
    summary.insert(0, "scenario", scenario)
    summary.insert(1, "scenario_label", SCENARIOS.get(scenario, scenario))
    scenario_heat_rows.append(summary)

heat_results = pd.concat(scenario_heat_rows, ignore_index=True) if scenario_heat_rows else pd.DataFrame()
if heat_results.empty:
    print("No exported heat-scenario networks available yet.")
else:
    display(heat_results)
    heat_results.to_csv(TABLE_DIR / "heat_supply_by_scenario.csv", index=False)


In [ ]:
if not heat_results.empty:
    supply = heat_results.pivot_table(
        index="scenario_label", columns="technology", values="heat_output_mwh", aggfunc="sum", fill_value=0
    ) / 1e3
    capacity = heat_results.pivot_table(
        index="scenario_label", columns="technology", values="capacity_mw_input", aggfunc="sum", fill_value=0
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    supply.plot(kind="bar", stacked=True, ax=axes[0])
    axes[0].set_ylabel("Heat supplied [GWh_th/yr]")
    axes[0].set_xlabel("")
    axes[0].set_title("Heat supply mix")
    axes[0].legend(fontsize=8)

    capacity.plot(kind="bar", stacked=True, ax=axes[1])
    axes[1].set_ylabel("Installed input capacity [MW]")
    axes[1].set_xlabel("")
    axes[1].set_title("Heat technology capacity")
    axes[1].legend(fontsize=8)
    for ax in axes:
        ax.tick_params(axis="x", rotation=30)
        ax.grid(True, alpha=0.25, axis="y")
    plt.tight_layout()
    if SAVE_FIGURES:
        fig.savefig(FIGURE_DIR / "heat_supply_and_capacity_scenarios.png", dpi=PAPER_DPI, bbox_inches="tight")
    plt.show()


## 15. Direct gas displacement, electricity for heat, and gas used for power

This section is deliberately **whole-system**.

A heat pump or electric boiler can strongly reduce direct gas use, but the extra electricity demand may increase gas-fired generation. Counting only the direct gas displaced would therefore overstate the fiscal saving when the electricity system remains gas-dependent.

For each scenario we track:

- useful heat supplied;
- direct gas input to heating technologies;
- electricity consumed by heating technologies;
- gas fuel input to electricity generation; and
- **total eligible gas = direct heat gas + power-sector gas**.

The exact set of gas-fired generator/link carriers can be edited in `GAS_POWER_CARRIERS`.


In [ ]:
def gas_power_fuel_input_mwh(n):
    """Estimate annual gas fuel input used for electricity generation.

    Generators: electrical output / efficiency.
    Links: positive p0 is treated as fuel input for carriers listed in GAS_POWER_CARRIERS.

    Returns np.nan only when the network structure cannot be evaluated.
    """
    total = 0.0
    found = False
    w = snapshot_weights(n)

    # Generator representation
    if hasattr(n, "generators") and not n.generators.empty and hasattr(n, "generators_t"):
        g = n.generators.copy()
        mask = g["carrier"].astype(str).str.lower().isin(GAS_POWER_CARRIERS)
        names = g.index[mask]
        if len(names) and hasattr(n.generators_t, "p"):
            p = n.generators_t.p.reindex(columns=names).fillna(0.0).clip(lower=0.0)
            for name in names:
                eff = pd.to_numeric(pd.Series([g.at[name, "efficiency"] if "efficiency" in g.columns else np.nan]), errors="coerce").iloc[0]
                if pd.notna(eff) and eff > 0:
                    total += float((p[name] * w).sum() / eff)
                    found = True

    # Link representation
    if hasattr(n, "links") and not n.links.empty and hasattr(n, "links_t"):
        l = n.links.copy()
        mask = l["carrier"].astype(str).str.lower().isin(GAS_POWER_CARRIERS)
        names = l.index[mask]
        if len(names) and hasattr(n.links_t, "p0"):
            p0 = n.links_t.p0.reindex(columns=names).fillna(0.0).clip(lower=0.0)
            total += float(p0.mul(w, axis=0).sum().sum())
            found = True

    return total if found else 0.0


def scenario_energy_metrics(heat_results, networks):
    if heat_results.empty:
        return pd.DataFrame()

    out = []
    electric_carriers = {"heat_pump", "electric_boiler", "district_heating"}
    gas_carriers = {"gas_boiler"}

    for (scenario, label), sub in heat_results.groupby(["scenario", "scenario_label"]):
        direct_gas = sub.loc[sub["carrier"].isin(gas_carriers), "input_mwh"].sum()
        electric_input = sub.loc[sub["carrier"].isin(electric_carriers), "input_mwh"].sum()
        heat_total = sub["heat_output_mwh"].sum()

        n = networks.get(scenario)
        power_gas = gas_power_fuel_input_mwh(n) if n is not None else np.nan

        out.append({
            "scenario": scenario,
            "scenario_label": label,
            "heat_output_mwh": heat_total,
            "direct_heat_gas_mwh": direct_gas,
            "electricity_for_heat_mwh": electric_input,
            "power_generation_gas_mwh": power_gas,
            "total_eligible_gas_mwh": direct_gas + power_gas if pd.notna(power_gas) else np.nan,
        })

    return pd.DataFrame(out)

energy_metrics = scenario_energy_metrics(heat_results, networks)

if not energy_metrics.empty:
    baseline_code = "GAS_BASE" if "GAS_BASE" in set(energy_metrics["scenario"]) else energy_metrics.iloc[0]["scenario"]
    base = energy_metrics.loc[energy_metrics["scenario"].eq(baseline_code)].iloc[0]

    energy_metrics["direct_gas_avoided_mwh"] = base["direct_heat_gas_mwh"] - energy_metrics["direct_heat_gas_mwh"]
    energy_metrics["power_gas_change_mwh"] = energy_metrics["power_generation_gas_mwh"] - base["power_generation_gas_mwh"]
    energy_metrics["net_eligible_gas_avoided_mwh"] = base["total_eligible_gas_mwh"] - energy_metrics["total_eligible_gas_mwh"]

    if GAS_LHV_KWH_PER_M3:
        energy_metrics["net_eligible_gas_avoided_m3"] = (
            energy_metrics["net_eligible_gas_avoided_mwh"] * 1000 / GAS_LHV_KWH_PER_M3
        )

    display(energy_metrics)
    energy_metrics.to_csv(TABLE_DIR / "scenario_energy_metrics.csv", index=False)


## 16. Net gas subsidy by transition scenario

This is the main policy comparison for the paper.

For scenario *s*:

**Net subsidy(s) = eligible direct-gas subsidy(s) + eligible power-sector gas subsidy(s)**

and relative to the gas baseline:

**Avoided subsidy(s) = Net subsidy(BAU) − Net subsidy(s)**

A positive value is a fiscal saving relative to BAU. A negative value means the scenario increases the gas-compensation burden.

The preferred implementation uses a documented subsidy intensity from `gas_subsidy.csv`, ideally monthly. The 2025 regional budget/ENAP totals are used only as external validation.


In [ ]:
def get_subsidy_intensity(subsidy_df, subsidy_baseline):
    """Return a transparent annual subsidy intensity.

    Priority:
      1) explicit subsidy_clp_per_mwh from the policy input table;
      2) explicit subsidy_clp_per_m3 converted with documented gas LHV;
      3) observed direct-gas baseline ratio (subsidy / fuel MWh).

    The function intentionally refuses to use the regional annual budget total as a
    unit factor because that would mix regional scope with Punta Arenas scenario scope.
    """
    if not subsidy_df.empty:
        s = subsidy_df.copy()
        if "year" in s.columns:
            s = s[s["year"].eq(ANALYSIS_YEAR)]

        if "subsidy_clp_per_mwh" in s.columns:
            vals = pd.to_numeric(s["subsidy_clp_per_mwh"], errors="coerce").dropna()
            if len(vals):
                return float(vals.mean()), "explicit gas_subsidy.csv CLP/MWh-fuel"

        if "subsidy_clp_per_m3" in s.columns and GAS_LHV_KWH_PER_M3:
            vals = pd.to_numeric(s["subsidy_clp_per_m3"], errors="coerce").dropna()
            if len(vals):
                factor = float(vals.mean()) * 1000.0 / GAS_LHV_KWH_PER_M3
                return factor, "gas_subsidy.csv CLP/m3 converted with documented gas LHV"

    if not subsidy_baseline.empty and "gas_mwh_fuel" in subsidy_baseline.columns:
        den = subsidy_baseline["gas_mwh_fuel"].sum()
        num = subsidy_baseline["subsidy_clp"].sum()
        if np.isfinite(num) and np.isfinite(den) and den > 0:
            return float(num / den), "observed direct-gas baseline ratio"

    if SUBSIDY_CLP_PER_MWH_DEFAULT is not None:
        return float(SUBSIDY_CLP_PER_MWH_DEFAULT), "manual default"

    if SUBSIDY_CLP_PER_M3_DEFAULT is not None and GAS_LHV_KWH_PER_M3:
        return float(SUBSIDY_CLP_PER_M3_DEFAULT) * 1000.0 / GAS_LHV_KWH_PER_M3, "manual default CLP/m3"

    return np.nan, "missing documented subsidy intensity"


def scenario_subsidy_accounting(energy_metrics, subsidy_df, subsidy_baseline):
    if energy_metrics.empty:
        return pd.DataFrame()

    intensity, intensity_source = get_subsidy_intensity(subsidy_df, subsidy_baseline)
    out = energy_metrics.copy()
    out["subsidy_intensity_clp_per_mwh_fuel"] = intensity
    out["subsidy_intensity_source"] = intensity_source

    if not np.isfinite(intensity):
        print(
            "A documented subsidy intensity is still required. "
            "Net-subsidy columns are left as NaN rather than inferred from the regional annual total."
        )
        out["net_gas_subsidy_clp"] = np.nan
        out["avoided_subsidy_clp"] = np.nan
        return out

    out["net_gas_subsidy_clp"] = out["total_eligible_gas_mwh"] * intensity

    baseline_code = "GAS_BASE" if "GAS_BASE" in set(out["scenario"]) else out.iloc[0]["scenario"]
    baseline_subsidy = float(out.loc[out["scenario"].eq(baseline_code), "net_gas_subsidy_clp"].iloc[0])
    out["avoided_subsidy_clp"] = baseline_subsidy - out["net_gas_subsidy_clp"]
    out["avoided_subsidy_pct"] = 100 * out["avoided_subsidy_clp"] / baseline_subsidy if baseline_subsidy else np.nan

    if CLP_PER_USD:
        out["net_gas_subsidy_usd"] = out["net_gas_subsidy_clp"] / CLP_PER_USD
        out["avoided_subsidy_usd"] = out["avoided_subsidy_clp"] / CLP_PER_USD

    return out

scenario_subsidy = scenario_subsidy_accounting(energy_metrics, subsidy, subsidy_baseline)

if not scenario_subsidy.empty:
    display(scenario_subsidy)
    scenario_subsidy.to_csv(TABLE_DIR / "net_gas_subsidy_by_scenario.csv", index=False)


## 17. Hourly operating comparison

Once model exports exist, compare the same winter/summer windows across heat technologies. This is useful for showing whether electrification creates new electric peaks and how strongly heat demand is seasonally concentrated.


In [ ]:
def plot_heat_operation_window(scenario, start, days=7):
    if scenario not in networks:
        raise KeyError(f"Scenario not loaded: {scenario}")
    n = networks[scenario]
    _, ts = heat_operation(n)
    start = pd.Timestamp(start)
    end = start + pd.Timedelta(days=days)

    supply = pd.DataFrame({
        TECH_LABELS.get(carrier, carrier): data["heat_output"]
        for carrier, data in ts.items()
    }).fillna(0)
    supply.index = pd.to_datetime(supply.index)
    supply = supply.loc[(supply.index >= start) & (supply.index < end)]

    fig, ax = plt.subplots(figsize=(10, 4))
    if not supply.empty:
        supply.plot.area(ax=ax, stacked=True, alpha=0.85)
    ax.set_ylabel("Heat supply [MW_th]")
    ax.set_xlabel("Time")
    ax.set_title(f"{SCENARIOS.get(scenario, scenario)} — heat operation")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

# Example after results exist:
# plot_heat_operation_window("HP", f"{ANALYSIS_YEAR}-07-01", days=7)


## 18. Paper summary table

This is the compact scenario-comparison table intended to underpin the paper's main argument.

The core columns are:

- heat supplied;
- direct gas consumption;
- electricity required for heat;
- gas consumed in power generation;
- total eligible gas;
- net gas subsidy;
- avoided subsidy relative to the gas baseline.

Whole-system cost and GHG emissions should be merged into the same table once their accounting is finalised, allowing the paper to compare **economic, fiscal, and climate trade-offs** consistently.


In [ ]:
def build_paper_summary(energy_metrics, scenario_subsidy):
    if energy_metrics.empty:
        return pd.DataFrame()

    out = energy_metrics.copy()

    if not scenario_subsidy.empty:
        subsidy_cols = [
            c for c in [
                "scenario",
                "subsidy_intensity_clp_per_mwh_fuel",
                "net_gas_subsidy_clp",
                "avoided_subsidy_clp",
                "avoided_subsidy_pct",
                "net_gas_subsidy_usd",
                "avoided_subsidy_usd",
            ]
            if c in scenario_subsidy.columns
        ]
        out = out.merge(scenario_subsidy[subsidy_cols], on="scenario", how="left")

    return out

paper_summary = build_paper_summary(energy_metrics, scenario_subsidy)

if not paper_summary.empty:
    display(paper_summary)
    paper_summary.to_csv(TABLE_DIR / "paper_summary.csv", index=False)


## 19. Export analysis metadata

Keep a machine-readable record of the key assumptions used for the figures generated in this notebook.


In [ ]:
analysis_metadata = {
    "analysis_year": ANALYSIS_YEAR,
    "city": CITY,
    "gas_lhv_kwh_per_m3": GAS_LHV_KWH_PER_M3,
    "gas_lhv_source": GAS_LHV_SOURCE,
    "clp_per_usd": CLP_PER_USD,
    "official_subsidy_budget_2025_clp": OFFICIAL_SUBSIDY_BENCHMARK_2025_CLP,
    "official_subsidy_actual_2025_usd": OFFICIAL_SUBSIDY_ACTUAL_2025_USD,
    "official_subsidy_sources": OFFICIAL_SUBSIDY_SOURCES,
    "gas_power_carriers": sorted(GAS_POWER_CARRIERS),
    "gas_hourly_sector_file": None if GAS_HOURLY_SECTOR_FILE is None else str(GAS_HOURLY_SECTOR_FILE),
    "heat_profile_file": None if HEAT_PROFILE_FILE is None else str(HEAT_PROFILE_FILE),
    "subsidy_file": None if SUBSIDY_FILE is None else str(SUBSIDY_FILE),
    "heat_assets_file": None if HEAT_ASSETS_FILE is None else str(HEAT_ASSETS_FILE),
    "costs_file": None if COSTS_FILE is None else str(COSTS_FILE),
    "scenario_labels": SCENARIOS,
}

with open(OUTPUT_ROOT / "analysis_metadata.json", "w", encoding="utf-8") as f:
    json.dump(analysis_metadata, f, indent=2, ensure_ascii=False)

print(json.dumps(analysis_metadata, indent=2, ensure_ascii=False))


## 20. Pending integration points

Before treating subsidy results as publication-ready:

- confirm the gas volume/energy convention and calorific value;
- finish the end-use split and useful-heat profile builder;
- build the documented subsidy-intensity input, preferably monthly, from the CNE compensation methodology rather than from the annual regional total;
- verify which modelled gas consumers are legally/policy-eligible for the compensatory mechanism;
- verify the carrier names used by gas-fired generators/links in the active PyPSA branch;
- finalise `heat_assets.csv` and `costs.csv` references;
- merge whole-system cost and GHG emissions into `paper_summary.csv`;
- use the 2025 regional values (CLP 65.629 billion budget; USD 64.6 million ENAP-reported compensation) as external validation only;
- add sensitivity around the subsidy intensity because the CNE mechanism is recalculated and can vary over time.

### Recommended main paper comparison

For each scenario report, at minimum:

`system_cost | emissions | direct_gas | electricity_for_heat | power_gas | total_eligible_gas | net_subsidy | avoided_subsidy`

This allows the analysis to distinguish a technology that merely **moves gas consumption from buildings to electricity generation** from one that genuinely reduces both fossil-gas dependence and the fiscal burden of the gas compensation.
